##Config

In [0]:
catalog_name = "cdl"

silver_schema_name = "bronze"
gold_schema_name = "gold"

storage_account_name = "datalakedevesh"
destination_container_name = "destination-cdl"

gold_base_path = (
    f"abfss://{destination_container_name}@{storage_account_name}.dfs.core.windows.net/gold/"
)

In [0]:
spark.sql(f"show tables in cdl.gold").select("database","tableName").display()

##Conversion of silver tables into df

In [0]:
def read_silver_tables(silver_table_name):
    return spark.read.table(silver_table_name)

# read_silver_tables("cdl.silver.product_master_silver").display()

## Delta tables write method 

In [0]:
def write_gold_table(df,table_name,write_mode = "overwrite"):

    target_table_name = f"{catalog_name}.{gold_schema_name}.{table_name}"
    target_table_path = f"{gold_base_path}/{table_name}/"

    print(f"Writing gold table {target_table_name}")
    print(f"Gold table path {target_table_path}")
    print(f"Write mode {write_mode}")

    df.write.format("delta").\
        mode(write_mode).\
            option("path",target_table_path).\
                option("overwriteSchema","true").\
                    saveAsTable(target_table_name)

    print(f"Successfully created {target_table_name} at {target_table_path}")

    return {
        "table_name":target_table_name,
        "table_path":target_table_path,
        "write_mode":write_mode,
        "status":"success"
    }

In [0]:
# df = read_silver_tables("cdl.silver.product_master_silver")
# # df.display()
# gold_write(df,"product_master_gold","overwrite")

##List of available gold tables

In [0]:
gold_tables = [
    "brand_weekly_performance",
    "hcp_engagement",
    "territory_performance",
    "call_rx_correlation"]

print(gold_tables)

##brand_weekly_performance table

In [0]:
from pyspark.sql.functions import *

#weekly rx aggregation
rx_silver_df = read_silver_tables("cdl.silver.rx_transactions_silver")
rx_weekly_df = rx_silver_df.alias("rx").join(
    read_silver_tables("cdl.silver.product_master_silver").alias("prod"),
    col("rx.product_id") == col("prod.product_id"),
    "left"
).groupBy(
    date_trunc("week",col("rx.rx_date")).cast("date").alias("week_start_date"),
    col("rx.product_id"),
    col("prod.product_name"),
    col("prod.brand"),
    col("prod.therapy_area")
).agg(
    sum("rx.prescription_count").alias("total_rx"),
    countDistinct("rx.hcp_id").alias("distinct_hcps"),
    max("rx.rx_date").alias("last_rx_date")
)

# rx_weekly_df.display()

#weekly call_aggregation

call_weekly_df = read_silver_tables("cdl.silver.call_activity_silver").alias("call").groupBy(
    date_trunc("week",col("call.call_date")).cast("date").alias("week_start_date"),
    col("call.product_id")
).agg(
    count('*').alias("total_calls"),
    sum(
        when(lower(col("call.call_status")) == "completed",1).otherwise(0)
    ).alias("completed_calls"),
    avg(col("call.duration_minutes")).alias("avg_duration"),
    max(col("call.call_date")).alias("last_call_date")
)

# call_weekly_df.display()

#final weekly performance

brand_weekly_performance_df = (
    rx_weekly_df.alias("rx")
        .join(
            call_weekly_df.alias("call"),
            (col("rx.week_start_date") == col("call.week_start_date")) &
            (col("rx.product_id") == col("call.product_id")),
            "full"
        )
        .select(
            coalesce(col("rx.week_start_date"), col("call.week_start_date")).alias("week_start_date"),
            coalesce(col("rx.product_id"), col("call.product_id")).alias("product_id"),

            col("rx.product_name"),
            col("rx.brand"),
            col("rx.therapy_area"),

            coalesce(col("rx.total_rx"), lit(0)).alias("total_prescriptions"),
            coalesce(col("rx.distinct_hcps"), lit(0)).alias("total_hcps_prescribing"),

            coalesce(col("call.total_calls"), lit(0)).alias("total_calls"),
            coalesce(col("call.completed_calls"), lit(0)).alias("completed_calls"),
            round(coalesce(col("call.avg_duration"), lit(0)), 2).alias("avg_call_duration"),

            col("rx.last_rx_date"),
            col("call.last_call_date"),

            current_timestamp().alias("gold_load_ts")
        )
)

brand_weekly_performance_df.display()

write_gold_table(brand_weekly_performance_df,"brand_weekly_performance","overwrite")



##HCP Engagement

In [0]:
# call metrics at hcp level
call_hcp_df = (
    read_silver_tables("cdl.silver.call_activity_silver")
        .groupBy("hcp_id")
        .agg(
            count("*").alias("total_calls"),

            sum(
                when(col("call_status") == "COMPLETED", 1).otherwise(0)
            ).alias("completed_calls"),

            sum(
                when(col("call_status") == "PLANNED", 1).otherwise(0)
            ).alias("planned_calls"),

            sum(
                when(col("call_status") == "CANCELLED", 1).otherwise(0)
            ).alias("cancelled_calls"),

            round(avg("duration_minutes"), 2).alias("avg_call_duration"),

            max("call_date").alias("last_call_date")
        )
)

#rx metrics per hcp
rx_hcp_df = (
    read_silver_tables("cdl.silver.rx_transactions_silver")
        .groupBy("hcp_id")
        .agg(
            sum("prescription_count").alias("total_prescriptions"),
            max("rx_date").alias("last_rx_date")
        )
)

#final hcp_engagement_df

hcp_engagement_df = (
    read_silver_tables("cdl.silver.hcp_master_silver").alias("hcp")
        .join(
            read_silver_tables("cdl.silver.territory_master_silver").alias("terr"),
            col("hcp.territory_id") == col("terr.territory_id"),
            "left"
        )
        .join(
            call_hcp_df.alias("call"),
            col("hcp.hcp_id") == col("call.hcp_id"),
            "left"
        )
        .join(
            rx_hcp_df.alias("rx"),
            col("hcp.hcp_id") == col("rx.hcp_id"),
            "left"
        )
        .select(
            col("hcp.hcp_id"),
            col("hcp.hcp_name"),
            col("hcp.specialty"),
            col("hcp.city"),
            col("hcp.state"),
            col("hcp.territory_id"),

            col("terr.territory_name"),
            col("terr.region"),
            col("terr.zone"),
            col("terr.sales_rep_id"),
            col("terr.sales_rep_name"),

            coalesce(col("call.total_calls"), lit(0)).alias("total_calls"),
            coalesce(col("call.completed_calls"), lit(0)).alias("completed_calls"),
            coalesce(col("call.planned_calls"), lit(0)).alias("planned_calls"),
            coalesce(col("call.cancelled_calls"), lit(0)).alias("cancelled_calls"),
            coalesce(col("call.avg_call_duration"), lit(0)).alias("avg_call_duration"),
            col("call.last_call_date"),

            coalesce(col("rx.total_prescriptions"), lit(0)).alias("total_prescriptions"),
            col("rx.last_rx_date"),

            current_timestamp().alias("gold_load_ts")
        )
)

hcp_engagement_df.display()

write_gold_table(hcp_engagement_df,"hcp_engagement","overwrite")


##territory performance

In [0]:
# ------------------------------------------------------------
# Step 1: Create HCP count per territory
# ------------------------------------------------------------

hcp_territory_df = (
    read_silver_tables("cdl.silver.hcp_master_silver")
        .groupBy("territory_id")
        .agg(
            countDistinct("hcp_id").alias("total_hcps")
        )
)


# ------------------------------------------------------------
# Step 2: Create call metrics per territory
# ------------------------------------------------------------

call_territory_df = (
    read_silver_tables("cdl.silver.call_activity_silver")
        .groupBy("territory_id")
        .agg(
            count("*").alias("total_calls"),

            sum(
                when(col("call_status") == "COMPLETED", 1).otherwise(0)
            ).alias("completed_calls"),

            sum(
                when(col("call_status") == "PLANNED", 1).otherwise(0)
            ).alias("planned_calls"),

            sum(
                when(col("call_status") == "CANCELLED", 1).otherwise(0)
            ).alias("cancelled_calls"),

            round(avg("duration_minutes"), 2).alias("avg_call_duration"),

            max("call_date").alias("last_call_date")
        )
)


# ------------------------------------------------------------
# Step 3: Create RX metrics per territory
# ------------------------------------------------------------

rx_territory_df = (
    read_silver_tables("cdl.silver.rx_transactions_silver")
        .groupBy("territory_id")
        .agg(
            sum("prescription_count").alias("total_prescriptions"),
            max("rx_date").alias("last_rx_date")
        )
)


# ------------------------------------------------------------
# Step 4: Build final territory_performance dataframe
# ------------------------------------------------------------

territory_performance_df = (
    read_silver_tables("cdl.silver.territory_master_silver").alias("terr")
        .join(
            hcp_territory_df.alias("hcp"),
            col("terr.territory_id") == col("hcp.territory_id"),
            "left"
        )
        .join(
            call_territory_df.alias("call"),
            col("terr.territory_id") == col("call.territory_id"),
            "left"
        )
        .join(
            rx_territory_df.alias("rx"),
            col("terr.territory_id") == col("rx.territory_id"),
            "left"
        )
        .select(
            col("terr.territory_id"),
            col("terr.territory_name"),
            col("terr.region"),
            col("terr.zone"),
            col("terr.sales_rep_id"),
            col("terr.sales_rep_name"),

            coalesce(col("hcp.total_hcps"), lit(0)).alias("total_hcps"),

            coalesce(col("call.total_calls"), lit(0)).alias("total_calls"),
            coalesce(col("call.completed_calls"), lit(0)).alias("completed_calls"),
            coalesce(col("call.planned_calls"), lit(0)).alias("planned_calls"),
            coalesce(col("call.cancelled_calls"), lit(0)).alias("cancelled_calls"),
            coalesce(col("call.avg_call_duration"), lit(0)).alias("avg_call_duration"),

            coalesce(col("rx.total_prescriptions"), lit(0)).alias("total_prescriptions"),

            col("call.last_call_date"),
            col("rx.last_rx_date"),

            current_timestamp().alias("gold_load_ts")
        )
)


# ------------------------------------------------------------
# Step 5: Preview final dataframe before writing
# ------------------------------------------------------------

territory_performance_df.display()


# ------------------------------------------------------------
# Step 6: Write Gold table
# ------------------------------------------------------------

write_gold_table(
    territory_performance_df,
    "territory_performance"
)

## call_rx correlation table

In [0]:
call_corr_df = (
    read_silver_tables("cdl.silver.call_activity_silver")
        .groupBy(
            date_trunc("week", col("call_date")).cast("date").alias("week_start_date"),
            "hcp_id",
            "product_id",
            "territory_id"
        )
        .agg(
            count("*").alias("weekly_calls"),

            sum(
                when(col("call_status") == "COMPLETED", 1).otherwise(0)
            ).alias("weekly_completed_calls"),

            sum(
                when(col("call_status") == "PLANNED", 1).otherwise(0)
            ).alias("weekly_planned_calls"),

            sum(
                when(col("call_status") == "CANCELLED", 1).otherwise(0)
            ).alias("weekly_cancelled_calls"),

            round(avg("duration_minutes"), 2).alias("weekly_avg_call_duration"),

            max("call_date").alias("last_call_date")
        )
)


# ------------------------------------------------------------
# Step 2: Create weekly RX metrics
# ------------------------------------------------------------

rx_corr_df = (
    read_silver_tables("cdl.silver.rx_transactions_silver")
        .groupBy(
            date_trunc("week", col("rx_date")).cast("date").alias("week_start_date"),
            "hcp_id",
            "product_id",
            "territory_id"
        )
        .agg(
            sum("prescription_count").alias("weekly_prescriptions"),
            max("rx_date").alias("last_rx_date")
        )
)


# ------------------------------------------------------------
# Step 3: Join weekly calls and weekly RX
# Full join is used because:
#   - Some HCP/product/week combinations may have calls but no RX
#   - Some HCP/product/week combinations may have RX but no calls
# ------------------------------------------------------------

call_rx_base_df = (
    call_corr_df.alias("call")
        .join(
            rx_corr_df.alias("rx"),
            (col("call.week_start_date") == col("rx.week_start_date")) &
            (col("call.hcp_id") == col("rx.hcp_id")) &
            (col("call.product_id") == col("rx.product_id")) &
            (col("call.territory_id") == col("rx.territory_id")),
            "full"
        )
)


# ------------------------------------------------------------
# Step 4: Enrich with HCP, Product, and Territory attributes
# ------------------------------------------------------------

call_rx_correlation_df = (
    call_rx_base_df
        .join(
            read_silver_tables("cdl.silver.hcp_master_silver").alias("hcp"),
            coalesce(col("call.hcp_id"), col("rx.hcp_id")) == col("hcp.hcp_id"),
            "left"
        )
        .join(
            read_silver_tables("cdl.silver.product_master_silver").alias("prod"),
            coalesce(col("call.product_id"), col("rx.product_id")) == col("prod.product_id"),
            "left"
        )
        .join(
            read_silver_tables("cdl.silver.territory_master_silver").alias("terr"),
            coalesce(col("call.territory_id"), col("rx.territory_id")) == col("terr.territory_id"),
            "left"
        )
        .select(
            coalesce(col("call.week_start_date"), col("rx.week_start_date")).alias("week_start_date"),

            coalesce(col("call.hcp_id"), col("rx.hcp_id")).alias("hcp_id"),
            col("hcp.hcp_name"),
            col("hcp.specialty"),
            col("hcp.city"),
            col("hcp.state"),

            coalesce(col("call.product_id"), col("rx.product_id")).alias("product_id"),
            col("prod.product_name"),
            col("prod.brand"),
            col("prod.therapy_area"),

            coalesce(col("call.territory_id"), col("rx.territory_id")).alias("territory_id"),
            col("terr.territory_name"),
            col("terr.region"),
            col("terr.zone"),
            col("terr.sales_rep_id"),
            col("terr.sales_rep_name"),

            coalesce(col("call.weekly_calls"), lit(0)).alias("weekly_calls"),
            coalesce(col("call.weekly_completed_calls"), lit(0)).alias("weekly_completed_calls"),
            coalesce(col("call.weekly_planned_calls"), lit(0)).alias("weekly_planned_calls"),
            coalesce(col("call.weekly_cancelled_calls"), lit(0)).alias("weekly_cancelled_calls"),
            coalesce(col("call.weekly_avg_call_duration"), lit(0)).alias("weekly_avg_call_duration"),

            coalesce(col("rx.weekly_prescriptions"), lit(0)).alias("weekly_prescriptions"),

            col("call.last_call_date"),
            col("rx.last_rx_date"),

            current_timestamp().alias("gold_load_ts")
        )
)


# ------------------------------------------------------------
# Step 5: Preview final dataframe before writing
# ------------------------------------------------------------

call_rx_correlation_df.display()


# ------------------------------------------------------------
# Step 6: Write Gold table
# ------------------------------------------------------------

write_gold_table(
    call_rx_correlation_df,
    "call_rx_correlation"
)

In [0]:
# spark.sql(f"show tables in cdl.silver").display()

# # spark.sql(f"select * from cdl.silver.call_activity_silver limit 10").display()

In [0]:
# spark.sql("""
# SELECT *
# FROM cdl.gold.brand_weekly_performance
# ORDER BY week_start_date, brand, product_id
# """).display()

In [0]:
# spark.sql("""
# DESCRIBE DETAIL cdl.gold.brand_weekly_performance
# """).select("name", "location", "format").display()